# AI-Based Customer Segmentation Using K-Means Clustering

This notebook demonstrates a complete end-to-end artificial intelligence project utilizing Unsupervised Learning to segment customers based on their purchasing behavior. We will use the **K-Means Clustering** algorithm applied to **RFM** (Recency, Frequency, Monetary) features.

## 1. Problem Definition & Data Collection

The objective is to automatically identify distinct customer groups so that targeted marketing campaigns can be launched. The dataset used is `Online Retail.xlsx`, containing transactional data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add src to path
sys.path.append('../src')

# Import custom modules
from preprocessing import load_data, clean_data
from feature_engineering import compute_rfm_features, compute_additional_features
from clustering import preprocess_for_clustering, find_optimal_k, train_final_model
from evaluation import evaluate_clusters, assign_segment_names, map_segments_to_data
from visualization import plot_elbow_and_silhouette, plot_rfm_distributions, plot_pca_clusters

# Load Data
df = load_data('../data/Online Retail.xlsx')
display(df.head())

## 2. Dataset Understanding

Let's check the basic statistics, missing values, and structure of the dataset.

In [ ]:
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")
print(f"Date Range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"Missing Values:\n{df.isnull().sum()}")
print(f"Cancelled Transactions: {df['InvoiceNo'].astype(str).str.startswith('C').sum()}")

## 3. Data Preprocessing

To ensure our AI model trains on high-quality data, we must handle missing values, duplicates, and cancelled transactions.

In [ ]:
clean_df = clean_data(df)
display(clean_df.describe())

## 4. Feature Engineering (RFM Analysis)

We convert transactional data into customer-level behavioral features: Recency, Frequency, and Monetary (RFM).

In [ ]:
rfm_df = compute_rfm_features(clean_df)
rfm_df = compute_additional_features(clean_df, rfm_df)
display(rfm_df.head())

## 5. Exploratory Analysis & Feature Transformation

Distance-based algorithms like K-Means are highly sensitive to skewness and scale. We plot the distributions, apply a log transformation, and scale the data using StandardScaler.

In [ ]:
plot_rfm_distributions(rfm_df, output_dir='../outputs/plots')
# Display the saved plot (optional in markdown/html)
from IPython.display import Image
display(Image(filename='../outputs/plots/rfm_distributions.png'))

In [ ]:
X_scaled_df, scaler = preprocess_for_clustering(rfm_df, features=['Recency', 'Frequency', 'Monetary'])
display(X_scaled_df.head())

## 6. Optimal K Selection

We determine the optimal number of clusters (K) using the **Elbow Method** (Inertia/WCSS) and **Silhouette Score**. We test K from 2 to 10.

In [ ]:
wcss, silhouette_scores = find_optimal_k(X_scaled_df, max_k=10)
plot_elbow_and_silhouette(wcss, silhouette_scores, output_dir='../outputs/plots')
display(Image(filename='../outputs/plots/optimal_k_evaluation.png'))

Based on the plots, we typically look for the 'elbow' point where the decrease in inertia slows down significantly, and where the silhouette score is reasonably high. We will choose **K=4** for this example as it often represents a good balance for customer segmentation (Champions, Loyal, At-Risk, Lost).

## 7. Train Final K-Means Model

We train the final model with K=4, assigning every customer to a specific cluster.

In [ ]:
rfm_df, kmeans_model = train_final_model(rfm_df, X_scaled_df, k=4, scaler=scaler)
display(rfm_df.head())

## 8. Customer Segment Interpretation

We evaluate the cluster averages to dynamically assign segment names.

In [ ]:
cluster_summary = evaluate_clusters(rfm_df)
cluster_summary, segment_map = assign_segment_names(cluster_summary)
rfm_df = map_segments_to_data(rfm_df, segment_map)

# Save outputs
os.makedirs('../outputs', exist_ok=True)
rfm_df.to_csv('../outputs/customer_segments.csv', index=False)
cluster_summary.to_csv('../outputs/cluster_summary.csv', index=False)

display(cluster_summary)

## 9. AI Visualization (PCA)

We use Principal Component Analysis (PCA) strictly to reduce the scaled 3D data into 2D for visualization purposes. The model itself was trained on the 3D data.

In [ ]:
plot_pca_clusters(X_scaled_df, rfm_df['Cluster'], segment_map, output_dir='../outputs/plots')
display(Image(filename='../outputs/plots/pca_clusters_2d.png'))

## 10. Business Insights & Recommendations

- **Champions**: Highly engaged, big spenders. Strategy: Loyalty programs, VIP perks.
- **Loyal Customers**: Consistent buyers. Strategy: Cross-selling and up-selling.
- **At-Risk Customers**: Good past behavior but haven't purchased recently. Strategy: Reactivation campaigns, discounts.
- **Lost Customers**: Low frequency, low spend, high recency. Strategy: Re-engagement campaigns if ROI is positive.

This completes the AI modeling phase. The models are saved and ready to be deployed in our interactive application.